# Case Study 2: Rare Perovskites — Ba₂FeMoO₆

This notebook is **Case Study 2** of the SKY paper.  
It demonstrates SKY's **recursive synthesis search** for a material that has
no direct synthesis recipe in the Materials Project database.

All cells run offline using pre-cached fixture data.


**Scientific Question:**  
For Ba₂FeMoO₆, which has no direct synthesis recipe in the MP database,
can SKY's recursive search find adaptable synthesis routes from chemically
similar double perovskites?

Ba₂FeMoO₆ is a double perovskite with a rock-salt ordered B-site sublattice
(Fe³⁺ and Mo⁵⁺/Mo³⁺ alternating on the octahedral site).  It exhibits
colossal magnetoresistance and half-metallic ferromagnetism, motivating
interest in its controlled synthesis, but detailed MP recipes are absent.


In [ ]:
import os
import sys
import json
import pathlib

# Add project root to path if running from notebooks/
PROJECT_ROOT = pathlib.Path(os.getcwd()).parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ.setdefault('OFFLINE_MODE', 'true')
OFFLINE_MODE = os.environ.get('OFFLINE_MODE', 'true').lower() in {'1', 'true', 'yes'}
print(f'OFFLINE_MODE = {OFFLINE_MODE}')
print(f'Project root : {PROJECT_ROOT}')

TARGET_FORMULA = 'Ba2FeMoO6'
print(f'Target material: {TARGET_FORMULA}')


## 1. Direct Recipe Search (Baseline)

We first attempt a direct recipe look-up to confirm that Ba₂FeMoO₆ is absent
from the Materials Project synthesis descriptions database.  This motivates
the recursive search in the sections that follow.


In [ ]:
from src.evaluation.mock_llm import MockSynthesisAgent

agent = MockSynthesisAgent()

direct_recipes = agent.get_synthesis_recipes_by_formula(TARGET_FORMULA)
print(f'Direct recipe search for {TARGET_FORMULA}:')
print(f'  Number of recipes found: {len(direct_recipes)}')

if len(direct_recipes) == 0:
    print(f'  Result: EMPTY — no direct synthesis recipe in the database.')
    print(f'  This confirms that recursive search is required.')
else:
    print(f'  Unexpected: {len(direct_recipes)} recipe(s) found. Showing first:')
    r = direct_recipes[0]
    if isinstance(r, dict):
        print(json.dumps(r, indent=4))


## 2. Similar Material Retrieval

Even without a direct recipe, SKY can retrieve chemically similar double
perovskites from the MAGPIE KNN index.  These neighbours serve as the
starting point for recipe adaptation.


In [ ]:
neighbors = agent.find_similar_materials_by_composition(TARGET_FORMULA, n_neighbors=8)

print(f'Top neighbours for {TARGET_FORMULA} (MAGPIE KNN):')
print(f'  {"Rank":<5} {"Formula":<22} {"Material ID":<16} {"Distance":>10} {"Confidence":>12}')
print('  ' + '-' * 70)
for rank, n in enumerate(neighbors, start=1):
    print(f'  {rank:<5} {n.formula:<22} {n.material_id:<16} {n.distance:>10.4f} {n.confidence:>12.4f}')

# Store for downstream cells
neighbor_formulas = [n.formula for n in neighbors]


## 3. Recursive Search Simulation

The `RecursiveSynthesisSearch` algorithm (Algorithm 3 in the Methods) performs
a depth-first search over the KNN graph.  Below we show the key logic
without triggering live MP API calls.

### Pseudocode recap

```
SEARCH(Ba2FeMoO6, depth=0, conf_threshold=1.0)
  ├─ No direct recipe → recurse into k=10 neighbours
  │
  ├─ Depth 1: BaFeO3    (conf ≥ 0.85)  → recipe found  ✓
  ├─ Depth 1: BaMoO4    (conf ≥ 0.82)  → recipe found  ✓
  ├─ Depth 1: SrFeMoO6  (conf ≥ 0.79)  → recipe found  ✓
  ├─ Depth 1: BaFeO2.5  (conf ≥ 0.75)  → recipe found  ✓
  └─ Depth 2: Ba2MoO5   (conf ≥ 0.68)  → recipe found via BaMoO4 ✓
```

**Depth-1 coverage:** materials reachable in one KNN hop (direct neighbours).  
**Depth-2 coverage:** materials reachable via two hops — expands the
effective neighbourhood from ~10 to ~100 candidates, recovering an additional
~18 % of materials with no depth-1 recipe (see Section 4 Coverage Analysis).


In [ ]:
# Illustrate the recursive search logic without API calls
# using the offline MockSynthesisAgent

print('Simulated recursive search (depth = 2, min_confidence = 0.70):')
print()

depth1_with_recipes = []
depth1_without_recipes = []

for n in neighbors:
    recipes = agent.get_synthesis_recipes_by_formula(n.formula)
    if recipes:
        depth1_with_recipes.append((n.formula, n.confidence, len(recipes)))
    else:
        depth1_without_recipes.append((n.formula, n.confidence))

print(f'Depth-1 neighbours WITH recipes  : {len(depth1_with_recipes)}')
for formula, conf, nrec in depth1_with_recipes:
    print(f'  {formula:<22}  conf={conf:.3f}  recipes={nrec}')

print()
print(f'Depth-1 neighbours WITHOUT recipes: {len(depth1_without_recipes)}')
for formula, conf in depth1_without_recipes:
    print(f'  {formula:<22}  conf={conf:.3f}  → will recurse at depth 2')

# Depth-2 expansion for the first no-recipe neighbour
if depth1_without_recipes:
    pivot_formula, _ = depth1_without_recipes[0]
    print(f'\nDepth-2 expansion from: {pivot_formula}')
    d2_neighbors = agent.find_similar_materials_by_composition(pivot_formula, n_neighbors=5)
    for n in d2_neighbors:
        recs = agent.get_synthesis_recipes_by_formula(n.formula)
        tag = f'recipes={len(recs)}' if recs else 'no recipes'
        print(f'  {n.formula:<22}  conf={n.confidence:.3f}  {tag}')


## 4. Identified Synthesis Routes from Neighbour Materials

We retrieve synthesis recipes from depth-1 neighbours — specifically the
Ba–Fe–O and Ba–Mo–O family — which are the closest analogues to Ba₂FeMoO₆
in MAGPIE space.


In [ ]:
# Retrieve recipes for depth-1 neighbours that have them
# If fixtures don't contain these exact formulas, show what IS available

candidate_formulas = ['BaFeO3', 'BaMoO4', 'SrFeMoO6', 'BaFeO2'] + neighbor_formulas[:4]

found_any = False
for formula in candidate_formulas:
    recipes = agent.get_synthesis_recipes_by_formula(formula)
    if not recipes:
        continue
    found_any = True
    print(f'\n--- {formula}: {len(recipes)} recipe(s) ---')
    for i, recipe in enumerate(recipes[:2], start=1):
        if isinstance(recipe, dict):
            stype = recipe.get('synthesis_type', 'unknown')
            temp  = recipe.get('temperature_celsius', 'N/A')
            para  = str(recipe.get('paragraph_string', ''))[:200]
            prec  = recipe.get('precursor_elements', [])
        else:
            stype = getattr(recipe, 'synthesis_type', 'unknown')
            temp  = getattr(recipe, 'temperature_celsius', 'N/A')
            para  = str(getattr(recipe, 'paragraph_string', ''))[:200]
            prec  = getattr(recipe, 'precursor_elements', [])
        print(f'  Recipe {i}: {stype}, {temp} °C, precursors: {prec}')
        print(f'    "{para}"')

if not found_any:
    # Show available keys in the fixture for diagnostic purposes
    available = list(agent._recipes.keys())[:10]
    print('No recipes found for candidate formulas in offline fixture.')
    print(f'Available fixture formulas (first 10): {available}')
    if available:
        formula = available[0]
        recipes = agent.get_synthesis_recipes_by_formula(formula)
        print(f'\nShowing recipes for fixture formula "{formula}" as representative output:')
        for i, recipe in enumerate(recipes[:1], start=1):
            if isinstance(recipe, dict):
                print(json.dumps(recipe, indent=2))


## 5. Adaptation Strategy

To synthesise Ba₂FeMoO₆ from a known BaFeO₃ recipe, the following element
substitutions and stoichiometry adjustments are required:

| Aspect | BaFeO₃ (source) | Ba₂FeMoO₆ (target) | Adaptation |
|--------|-----------------|---------------------|------------|
| Formula | BaFeO₃ | Ba₂FeMoO₆ | Ba: 1→2, add Mo, O: 3→6 |
| Elements added | — | Mo | Add MoO₃ or (NH₄)₂MoO₄ precursor |
| Elements removed | — | — | None |
| Ba stoichiometry | 1 | 2 | Double Ba source (BaCO₃) amount |
| Oxygen content | 3 | 6 | Adjust atmosphere (reducing vs oxidising) |
| Typical method | Solid-state, 900 °C | Solid-state / H₂ reduction | Reduce to obtain Fe²⁺/Mo⁵⁺ |
| Special notes | Air atmosphere | Forming gas (5 % H₂/N₂) | Double-perovskite requires B-site ordering |


In [ ]:
from src.recursive_synthesis import RecursiveSynthesisSearch
from src.evaluation.mock_llm import MockSynthesisAgent as _MockAgent

# Instantiate with the mock agent — no MP API calls will be made
_mock = _MockAgent()

# We call _calculate_adaptation directly (static-style method)
# This only uses pymatgen.Composition and requires no API
rss = RecursiveSynthesisSearch.__new__(RecursiveSynthesisSearch)

adaptation = rss._calculate_adaptation('Ba2FeMoO6', 'BaTiO3')
print('_calculate_adaptation("Ba2FeMoO6", "BaTiO3"):')
print(json.dumps(adaptation, indent=2))

print()
adaptation2 = rss._calculate_adaptation('Ba2FeMoO6', 'BaFeO3')
print('_calculate_adaptation("Ba2FeMoO6", "BaFeO3"):')
print(json.dumps(adaptation2, indent=2))


## 6. Coverage Analysis

### Direct KNN vs Recursive Search — coverage benefit

For materials absent from the MP synthesis database (as Ba₂FeMoO₆ is), the
recursive search extends coverage at the cost of additional hops.  The
table below summarises the observed trade-off from the ablation study
(`src.evaluation.ablation.run_recursive_ablation`):

| Method | Coverage (≥1 recipe) | Mean SRO@5 (covered) |
|--------|---------------------|----------------------|
| Direct KNN (depth=0) | 71 % | 0.58 |
| Recursive (depth≤1) | 85 % | 0.54 |
| Recursive (depth≤2) | 89 % | 0.51 |

**Key finding:** Depth-2 recursive search recovers an additional **~18 %** of
rare materials that have no direct KNN recipe, at a modest cost of ~7 % lower
SRO quality (because depth-2 recipes are chemically more distant).

> **Note:** The full ablation requires the MP embedding HDF5 files and
> is run via `src.evaluation.ablation.run_recursive_ablation`.  The numbers
> above are from the paper's Table 2.
